# 04 · Pretrained Model + The Big Evaluation 🏆  ·  *Role D*

**Your mission has two halves:**
1. Build the **transfer-learning** model (a pretrained ResNet18 that already knows generic shapes).
2. Run the headline experiment: test both models on a **disaster they've never seen**, compare
   them fairly, and draw a real **damage map**. This is where the project's story comes together. 🎬

In [ ]:
import sys, os; sys.path.append(os.path.abspath('..'))
from src import config, data_utils, models, train_utils
import torch, numpy as np
from torch.utils.data import DataLoader
from pathlib import Path
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training on:', device)

### The model: a pretrained backbone, shared across both branches 🧠
Instead of learning from zero, we start from a ResNet18 trained on millions of images. We
**freeze** its early layers (generic edges/textures) and only fine-tune the later ones + a small
head. Same siamese trick as notebook 03, just smarter eyes.

In [ ]:
model = models.SiamesePretrainedCNN(num_classes=config.NUM_CLASSES,
                                    backbone='resnet18', freeze_until=6).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
print('Fine-tuning', round(trainable, 2), 'M parameters')

### Train on the seen disasters (same recipe as notebook 03, for a fair fight) 🥊

In [ ]:
train_dl = DataLoader(data_utils.DamageDataset('train', augment=True),  batch_size=config.BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(data_utils.DamageDataset('val',   augment=False), batch_size=config.BATCH_SIZE)
w_path = Path(config.DATA_DIR)/'processed'/'class_weights.npy'
class_weights = torch.tensor(np.load(w_path), dtype=torch.float32) if w_path.exists() else None

history = train_utils.train_model(model, train_dl, val_dl, device, epochs=config.EPOCHS,
                                  class_weights=class_weights, ckpt_path='../checkpoints/pretrained.pt')
train_utils.plot_history(history)

## 🎯 The key result: test on a disaster nobody trained on
Now we load **both** trained models and run them on `config.HELDOUT_DISASTER` (wildfires).
If accuracy drops a lot vs the seen disasters, that's the **transfer gap** — and it's exactly
the finding the project is about. Compare them honestly.

In [ ]:
test_dl = DataLoader(data_utils.DamageDataset('test', augment=False), batch_size=config.BATCH_SIZE)

# load the from-scratch model Role C trained
scratch = models.SiameseScratchCNN(config.NUM_CLASSES).to(device)
scratch.load_state_dict(torch.load('../checkpoints/scratch.pt', map_location=device))

print('=== FROM-SCRATCH on the held-out disaster ===')
train_utils.evaluate(scratch, test_dl, device, class_names=config.DAMAGE_CLASSES)
print('\n=== PRETRAINED on the held-out disaster ===')
train_utils.evaluate(model, test_dl, device, class_names=config.DAMAGE_CLASSES)

### Put the numbers in one table 📋
👉 *TODO(Role D):* make a small DataFrame — rows = {scratch, pretrained} × {seen, held-out},
columns = accuracy, macro-F1, destroyed-recall. One glance should tell the whole story.

💡 *Optional bonus (easy points):* add just a handful of labeled wildfire patches to training
(few-shot) and show how much the gap shrinks.

In [ ]:
# TODO(Role D): build the comparison DataFrame from the evaluate() outputs above.

## 🗺️ Deployment: draw a damage map of a real event
Take one held-out event tile, predict every building with the better model, and color each
building footprint by its predicted damage (green = no-damage … red = destroyed) over the
post-disaster image. This is the picture that goes on the slide. 🖼️

👉 *TODO(Role D):* also note where it's wrong — honesty about failures earns points.

In [ ]:
# TODO(Role D): tile inference -> colored polygon overlay -> save the figure.
# 1. pick a held-out event tile
# 2. crop all its building patches (reuse Role A's crop_patch)
# 3. predict with the better model
# 4. draw each footprint colored by predicted class